## step 1: Read all the data from the source systems

#### 1. Stock price

In [0]:
df = spark.read.option("inferSchema", "true").option("header", "true").csv("/Volumes/datalakehouse/bronze/source_system/S&P_500_Stock_price/all_stocks_5yr.csv")


In [0]:
df.display()

In [0]:
df.write.mode("overwrite").saveAsTable("datalakehouse.bronze.bronze_stock_prices")

### 2. Company List

#### Let's read all the other files in the S&P500 company list at once using a simple loop
#### Then write the data to the Bronze layer without modification. This is very important because we want to preserve the raw data from the system so that if something breaks later in our pipeline we can always have a version to go back

In [0]:
# Let's write a for loop to read the data into a table
datasets = ['constituents.csv','sector.csv']

In [0]:
for _ in datasets:
    df = spark.read.option("inferSchema", "true").option("header", "true").csv(
        f"/Volumes/datalakehouse/bronze/source_system/S&P 500 Companies List/{_}"
    )
     # Clean column names: replace spaces and special chars with underscores
    df = df.toDF(*[c.strip().lower().replace(" ", "_").replace("-", "_").replace("/", "_").replace("&", "and") for c in df.columns])
    df.write.mode("overwrite").saveAsTable(
        f"datalakehouse.bronze.bronze_{_.replace('.csv','')}"
    )
    df.display()

### 3. Company fundamentals (ETF & Stocks)

In [0]:
# Now for the last folder: Company fundamentals
df = spark.read.option("inferSchema", "true").option("header", "true").csv(
    "/Volumes/datalakehouse/bronze/source_system/Company_fundamentals/symbols_valid_meta.csv"
    )
df = df.toDF(*[c.strip().lower().replace(" ", "_").replace("-", "_").replace("/", "_").replace("&", "and") for c in df.columns])
df.write.mode("overwrite").saveAsTable("datalakehouse.bronze.bronze_symbols_valid_meta")

In [0]:
%sql 
select count (*) as total_rows from datalakehouse.bronze.bronze_constituents